# DeceptEnv â€” RL training (TRL GRPO, Colab-ready)

This notebook trains an LLM Agent to deceive a frozen Detective inside the **DeceptEnv** OpenEnv environment, using **Hugging Face TRL's GRPOTrainer**.

It is designed to run end-to-end on a free Colab T4 (or locally on CPU with the 0.5B model). The same pipeline is the source of the `suspicion_curve.png` and `reward_curve.png` artefacts in the README.

## 0. Install + clone
Skip the install line if you've already `pip install -r requirements.txt` in this environment.

In [ ]:
!pip install --quiet 'transformers>=4.44' 'trl>=0.10' 'peft>=0.12' 'accelerate>=0.33' \
                    'datasets>=2.20' 'fastapi>=0.115' 'uvicorn[standard]>=0.30' \
                    'httpx>=0.27' 'matplotlib>=3.8' 'numpy>=1.26'

!git clone https://huggingface.co/spaces/Jaisharma7/DeceptEnv /content/DeceptEnv
%cd /content/DeceptEnv

import sys, pathlib
ROOT = pathlib.Path('.').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print('cwd:', ROOT)

from google.colab import userdata
from huggingface_hub import login
login(token=userdata.get('HF_TOKEN'))


## 1. Boot the DeceptEnv server in the background

TRL's reward function will call the live HTTP server. The server uses the **mock Detective** by default â€” no API keys required.

In [ ]:
import os, time, httpx
BASE_URL = 'https://jaisharma7-deceptenv.hf.space'

print(f'Checking connection to deployed HF Space: {BASE_URL}...')
for _ in range(60):
    try:
        r = httpx.get(f'{BASE_URL}/healthz', timeout=5.0)
        if r.status_code == 200:
            print('Server is up!', r.json())
            break
    except Exception as e:
        print('Waiting for space to wake up...')
    time.sleep(2.0)
else:
    raise RuntimeError('Deployed DeceptEnv server at HF Space failed to start or is asleep. Check your Space logs.')


## 2. Build a single-turn deception dataset

Each example is a fresh `reset()` observation rendered as a chat-formatted prompt. The reward function will reset the env to the same scenario seed and execute one `step()` per completion to produce the rubric reward.

In [ ]:
from client import DeceptEnvClient
from training.agent_policy import format_agent_prompt
from datasets import Dataset

DATASET_SIZE = 256

_probe = DeceptEnvClient(BASE_URL, env_id='probe')
rows = []
scenarios = _probe.scenarios()
for i in range(DATASET_SIZE):
    sid = scenarios[i % len(scenarios)]
    obs, info = _probe.reset(seed=10_000 + i, scenario_id=sid)
    sys_p, user_p = format_agent_prompt(obs)
    rows.append({
        'prompt': [
            {'role': 'system', 'content': sys_p},
            {'role': 'user',   'content': user_p},
        ],
        'scenario_id': sid,
        'seed': 10_000 + i,
    })
_probe.close()
ds = Dataset.from_list(rows)
print(ds, '\nexample prompt[1].content[:200]:', ds[0]['prompt'][1]['content'][:200], '...')

## 3. Reward function (the rubric, served over HTTP)

TRL passes us `completions` plus any extra dataset columns we asked for. We re-`reset` the env to the matching seed/scenario and `step` once with the completion. The returned reward is the env's rubric: $\Delta$suspicion \* 2 with $-50/-15$ guardrail penalties.

In [ ]:
from threading import Lock
_score_clients = {}
_pool_lock = Lock()

def _client_for(env_id):
    with _pool_lock:
        c = _score_clients.get(env_id)
        if c is None:
            c = DeceptEnvClient(BASE_URL, env_id=env_id)
            _score_clients[env_id] = c
        return c

def deceptenv_reward(prompts, completions, scenario_id, seed, **_):
    """Score each completion by stepping a fresh env at the row's (scenario_id, seed)."""
    rewards = []
    for i, (sid, sd, comp) in enumerate(zip(scenario_id, seed, completions)):
        # GRPO sometimes hands us the chat-formatted message list â€” flatten.
        if isinstance(comp, list):
            comp = comp[-1].get('content', '') if comp and isinstance(comp[-1], dict) else str(comp)
        env_id = f'reward-{i}'
        client = _client_for(env_id)
        client.reset(seed=int(sd), scenario_id=str(sid))
        result = client.step(str(comp))
        rewards.append(float(result.reward))
    return rewards

## 4. Load the model (LoRA + small instruct base)

Defaults to a tiny 0.5B Qwen for fast laptop iteration. On Colab T4 swap in `Qwen/Qwen2.5-1.5B-Instruct` or, with `bitsandbytes`/Unsloth, `meta-llama/Meta-Llama-3-8B-Instruct`.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig
from trl import GRPOConfig, GRPOTrainer

MODEL_NAME = os.environ.get('DECEPTENV_TRAIN_MODEL', 'Qwen/Qwen2.5-0.5B-Instruct')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=dtype)

peft_config = LoraConfig(
    r=8, lora_alpha=16, lora_dropout=0.05, bias='none',
    task_type='CAUSAL_LM',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
)

## 5. GRPO config and trainer

In [ ]:
grpo_cfg = GRPOConfig(
    output_dir='./runs/grpo',
    learning_rate=5e-5,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=2,
    num_generations=4,           # group size for GRPO advantages
    max_completion_length=96,
    num_train_epochs=1,
    logging_steps=2,
    save_steps=20,
    bf16=torch.cuda.is_available(),
    report_to='none',
    temperature=0.9,
)
trainer = GRPOTrainer(
    model=model,
    args=grpo_cfg,
    train_dataset=ds,
    reward_funcs=[deceptenv_reward],
    peft_config=peft_config,
    processing_class=tokenizer,
)
trainer.train()

## 6. Plot suspicion + reward curves from training history

We extract reward and suspicion summaries from the trainer's log history. (For multi-turn evolution see `evaluation/evaluate.py`.)

In [ ]:
from analytics.plotter import plot_reward_curve
import numpy as np, matplotlib.pyplot as plt, json

history = trainer.state.log_history
rewards = [h['reward'] for h in history if 'reward' in h]
if rewards:
    plot_reward_curve(rewards, './runs/grpo/reward_curve.png')
    print('saved ./runs/grpo/reward_curve.png')
    plt.figure(figsize=(8,4)); plt.plot(rewards); plt.title('GRPO reward'); plt.grid(alpha=.3); plt.show()
else:
    print('no reward in trainer history yet â€” run training for more steps')

## 7. Multi-turn evaluation (the headline number)

Single-turn GRPO is the optimisation; the evaluation runs full 10-turn episodes â€” that's where the multi-turn deception score is measured. The same code is in `evaluation/evaluate.py`.

In [ ]:
from training.agent_policy import HFCausalAgent, GenerationConfig
from training.rollout import run_episode
from analytics.plotter import summarise_episodes, plot_suspicion_curve

policy = HFCausalAgent(MODEL_NAME, model=trainer.model, tokenizer=tokenizer,
                       generation=GenerationConfig(temperature=0.7, top_p=0.95, max_new_tokens=80))
client = DeceptEnvClient(BASE_URL, env_id='eval')
summaries = []
susp_per_ep = []
for i in range(20):
    roll = run_episode(client, policy.act, seed=99_000 + i,
                       scenario_id=scenarios[i % len(scenarios)])
    summaries.append(roll.summary)
    susp_per_ep.append(roll.summary.final_suspicion)
    print(f"ep {i:02d} susp={roll.summary.final_suspicion:3d} "
          f"reward={roll.summary.total_reward:+.1f} "
          f"end={roll.summary.terminal_reason}")
client.close()
agg = summarise_episodes(summaries)
print('\nAGG:', json.dumps(agg, indent=2))
plot_suspicion_curve(susp_per_ep, './runs/grpo/suspicion_curve.png',
                     title='Final suspicion across post-train eval episodes',
                     label='final suspicion')